# Pregunta 2 — Análisis factorial exploratorio

**Métodos Cuantitativos de la Investigación en Negocios — Tarea 2 (2026)**

In [1]:
# Instalación condicional de dependencias para Google Colab.
# En ejecución local no instala nada si los paquetes ya están disponibles.
import importlib, subprocess, sys

def instalar_si_falta(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f'Instalando {package_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

instalar_si_falta('pyreadstat', 'pyreadstat')
instalar_si_falta('factor_analyzer', 'factor-analyzer')

## Enunciado de la pregunta 2

**2.** La Universidad de Desarrollo está permanentemente tratando de tener los mejores profesores entre sus filas. Como parte de este esfuerzo, la UDD realizó un esfuerzo por analizar el cuestionario basado en la teoría de Blend asociada a los profesores de métodos de investigación. Esta teoría predice que los buenos profesores de métodos cuantitativos poseen cuatro características fundamentales: (1) gran amor por las estadísticas, (2) entusiasmo por el diseño de experimentos, (3) amor por la enseñanza, y (4) una completa ausencia de habilidades interpersonales normales. Estas características pueden estar correlacionadas entre ellas.

El cuestionario “Enseñanza de Experimentos para las Ciencias Sociales” (EECS) ya existe, pero la universidad mejoró este cuestionario y se convirtió en el EECS-R (“Enseñanza de Experimentos para las Ciencias Sociales - Revisado”). Este cuestionario revisado fue enviado y contestado satisfactoriamente por 239 profesores de métodos de investigación alrededor del mundo para evaluar si la teoría de Blend es correcta.

El cuestionario (preguntas, escala y respuestas) se encuentra en el archivo `EECS-R.sav`. Realice un Análisis Factorial Exploratorio y determine si se cumple la teoría de Blend.

## Respuesta

La pregunta se resuelve verificando primero si los datos son adecuados para análisis factorial, determinando cuántos factores retener, interpretando las cargas factoriales y evaluando si la estructura empírica coincide con las cuatro dimensiones propuestas por la teoría de Blend.


In [3]:
from pathlib import Path
from urllib.request import urlretrieve, urlopen
import ssl
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

# Base requerida para este notebook.
# Funciona localmente y también en Google Colab abierto desde GitHub.
REQUIRED_DATA_FILE = 'EECS-R.sav'
RAW_DATA_BASE = 'https://raw.githubusercontent.com/sebabecerra/Metodos-Cuantitativos-DEN/main/Tarea2/data'

def encontrar_o_descargar_base(filename):
    candidatas = [
        Path.cwd() / 'data',                 # ejecución desde Tarea2
        Path.cwd().parent / 'data',          # ejecución desde Tarea2/notebooks
        Path.cwd() / 'Tarea2' / 'data',      # ejecución desde raíz del repo o Colab
        Path.cwd()                           # respaldo: base junto al notebook
    ]
    for ruta in candidatas:
        if (ruta / filename).exists():
            return ruta

    # Si Colab no tiene la carpeta del repo clonada, se descarga la base desde GitHub.
    destino_dir = Path.cwd() / 'Tarea2' / 'data'
    destino_dir.mkdir(parents=True, exist_ok=True)
    destino = destino_dir / filename
    url = f'{RAW_DATA_BASE}/{filename}'
    print(f'Base {filename} no encontrada localmente. Descargando desde GitHub...')
    try:
        urlretrieve(url, destino)
    except Exception as error:
        # Respaldo útil en algunos entornos locales con problemas de certificados SSL.
        # En Colab normalmente no se activa, pero evita fallas si Python no encuentra certificados.
        print('Descarga estándar falló; intentando respaldo SSL...')
        contexto = ssl._create_unverified_context()
        with urlopen(url, context=contexto) as respuesta, open(destino, 'wb') as archivo:
            archivo.write(respuesta.read())
    return destino_dir

DATA_DIR = encontrar_o_descargar_base(REQUIRED_DATA_FILE)
print('Directorio de datos:', DATA_DIR)
print('Base usada:', DATA_DIR / REQUIRED_DATA_FILE)


Directorio de datos: /Users/sbc/projects/Metodos-Cuantitativos-DEN/Tarea2/data
Base usada: /Users/sbc/projects/Metodos-Cuantitativos-DEN/Tarea2/data/EECS-R.sav


## 1. Objetivo y decisiones analíticas

Se evalúa si los 28 ítems del EECS-R reproducen cuatro dimensiones correlacionadas propuestas por la teoría de Blend: amor por la estadística, entusiasmo por el diseño experimental, amor por la enseñanza y ausencia de habilidades interpersonales.

Se usa **análisis factorial exploratorio de factores comunes**, extracción de máxima verosimilitud y rotación oblicua Oblimin, porque las dimensiones pueden correlacionarse. Los datos perdidos —solo nueve respuestas— se imputan con la mediana del ítem.

In [3]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
import inspect
import sklearn.utils.validation as sk_validation
import factor_analyzer.factor_analyzer as fa_module

# Compatibilidad: algunas versiones nuevas de scikit-learn reemplazan
# force_all_finite por ensure_all_finite. factor_analyzer todavía usa
# force_all_finite en ciertos entornos, por eso se traduce el argumento.
_original_check_array = sk_validation.check_array
def _check_array_compat(*args, **kwargs):
    if (
        'force_all_finite' in kwargs
        and 'force_all_finite' not in inspect.signature(_original_check_array).parameters
    ):
        kwargs['ensure_all_finite'] = kwargs.pop('force_all_finite')
    return _original_check_array(*args, **kwargs)

fa_module.check_array = _check_array_compat

# Chequeo inicial de la base.
# Se carga la base completa; no se filtran observaciones ni se toma una submuestra.
df, meta = pyreadstat.read_sav(DATA_DIR/'EECS-R.sav', apply_value_formats=False)
items = [f'q{i}' for i in range(1, 29)]
X = df[items].copy()

print('Archivo usado:', DATA_DIR/'EECS-R.sav')
print('Número de observaciones:', len(df))
print('Número de ítems analizados:', len(items))
print('Dimensión de la matriz de ítems:', X.shape)
faltantes = int(X.isna().sum().sum())
print('Datos perdidos totales en los ítems:', faltantes)
print(f'Porcentaje de datos perdidos: {100 * faltantes / X.size:.3f}%')
print('\nDatos perdidos por ítem:')
print(X.isna().sum().to_string())
print('\nDescriptivos básicos de los ítems:')
print(X.describe().T[['mean', 'std', 'min', 'max']].round(3).to_string())
print('\nEtiquetas de ítems:')
for c in items:
    print(f'{c}: {meta.column_names_to_labels[c]}')

# Imputación por mediana debido a la baja proporción de datos perdidos.
X = X.apply(lambda s: s.fillna(s.median()))
print('\nDatos perdidos después de imputar por mediana:', int(X.isna().sum().sum()))


Archivo usado: /Users/sbc/projects/Metodos-Cuantitativos-DEN/Tarea2/data/EECS-R.sav
Número de observaciones: 239
Número de ítems analizados: 28
Dimensión de la matriz de ítems: (239, 28)
Datos perdidos totales en los ítems: 9
Porcentaje de datos perdidos: 0.134%

Datos perdidos por ítem:
q1     0
q2     1
q3     1
q4     0
q5     0
q6     0
q7     0
q8     0
q9     0
q10    0
q11    1
q12    2
q13    0
q14    0
q15    1
q16    0
q17    0
q18    1
q19    1
q20    0
q21    1
q22    0
q23    0
q24    0
q25    0
q26    0
q27    0
q28    0

Descriptivos básicos de los ítems:
      mean    std  min  max
q1   4.720  1.412  1.0  7.0
q2   4.782  1.391  1.0  7.0
q3   4.929  1.158  1.0  7.0
q4   4.791  1.243  1.0  7.0
q5   3.406  1.315  1.0  7.0
q6   5.276  1.387  1.0  7.0
q7   4.987  1.102  1.0  7.0
q8   5.213  1.277  1.0  7.0
q9   5.314  1.249  1.0  7.0
q10  4.444  1.454  1.0  7.0
q11  4.080  1.545  1.0  7.0
q12  3.865  1.423  1.0  7.0
q13  5.431  1.304  2.0  7.0
q14  4.895  1.575  1.0  7.0
q15

## 2. Factorabilidad: Bartlett y KMO

Bartlett contrasta que la matriz de correlaciones sea identidad. Se requiere $p<0.05$. KMO cuantifica varianza común; valores superiores a 0.80.

La prueba se calcula como $\chi^2=-\bigl(n-1-\tfrac{2p+5}{6}\bigr)\ln|R|$, con $gl=p(p-1)/2$, donde $n$ es el número de casos, $p$ el número de ítems y $|R|$ el determinante de la matriz de correlaciones. La regla de decisión es: se rechaza $H_0$ (matriz identidad) si el estadístico supera el valor crítico $\chi^2_{0.95,gl}$, equivalente a $p<0.05$.

In [7]:
chi,p=calculate_bartlett_sphericity(X); _,kmo=calculate_kmo(X)
print(f'Bartlett: chi-cuadrado={chi:.4f}, gl={28*27//2}, ' + ('p<0.001' if p<0.001 else f'p={p:.4g}'))
print(f'KMO global={kmo:.4f}')

# Verificación manual de Bartlett: chi2 = -(n-1-(2p+5)/6)·ln|R|.
# Reproducir la fórmula a mano documenta el cálculo y valida la función de la librería.
lnR = np.linalg.slogdet(X.corr().to_numpy())[1]
chi_manual = -(len(X)-1-(2*28+5)/6)*lnR
print(f'\nln|R| = {lnR:.4f}')
print(f'chi2 manual = -(239-1-61/6)·({lnR:.4f}) = {chi_manual:.4f}  (coincide con la función)')
print(f'chi2 crítico (95%, gl=378) = {stats.chi2.ppf(.95,378):.4f} -> como {chi:.2f} > {stats.chi2.ppf(.95,378):.2f}, se rechaza H0')

Bartlett: chi-cuadrado=3045.3894, gl=378, p<0.001
KMO global=0.8952

ln|R| = -13.3667
chi2 manual = -(239-1-61/6)·(-13.3667) = 3045.3894  (coincide con la función)
chi2 crítico (95%, gl=378) = 424.3342 -> como 3045.39 > 424.33, se rechaza H0


## 3. Número de factores

Para decidir cuántos factores retener se combinan tres criterios:

1. **Autovalores observados:** indican cuánta varianza explica cada dimensión inicial.
2. **Gráfico de sedimentación:** permite observar el punto donde la curva comienza a aplanarse.
3. **Análisis paralelo:** compara los autovalores observados con los que aparecerían por azar en matrices aleatorias del mismo tamaño.

El análisis paralelo es especialmente importante porque el criterio de autovalor mayor que 1 tiende a sobreextraer factores. En esta aplicación, el cuarto factor queda muy cerca del percentil 95 aleatorio; por eso se revisa también la media de autovalores aleatorios y la coherencia teórica de cuatro dimensiones.

In [9]:
R=X.corr().to_numpy(); eig=np.linalg.eigvalsh(R)[::-1]
rng=np.random.default_rng(12345)
rand=[]
for _ in range(1000):
    Rr=np.corrcoef(rng.normal(size=X.shape),rowvar=False)
    rand.append(np.linalg.eigvalsh(Rr)[::-1])
rand=np.array(rand)
rand95=np.percentile(rand,95,axis=0)
randmean=rand.mean(axis=0)
print('Primeros 8 autovalores observados:',np.round(eig[:8],3))
print('Percentil 95 aleatorio:',np.round(rand95[:8],3))
print('Media aleatoria:',np.round(randmean[:8],3))
print('Factores retenidos por percentil 95:',int((eig>rand95).sum()))
print('Factores retenidos por media aleatoria:',int((eig>randmean).sum()))
print(f'Cuarto autovalor observado = {eig[3]:.3f}; percentil 95 aleatorio = {rand95[3]:.3f}; media aleatoria = {randmean[3]:.3f}')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,5))
xs=np.arange(1,len(items)+1)
ax.plot(xs,eig,marker='o',label='Autovalores observados',linewidth=2)
ax.plot(xs,rand95,marker='s',label='Percentil 95 aleatorio',linewidth=1.5)
ax.plot(xs,randmean,linestyle='--',label='Media aleatoria',linewidth=1.5)
ax.axvline(4,color='gray',linestyle=':',label='4 factores retenidos')
ax.set_xlabel('Número de factor')
ax.set_ylabel('Autovalor')
ax.set_title('Análisis paralelo y gráfico de sedimentación')
ax.set_xlim(1,12)
ax.grid(alpha=.25)
ax.legend()
plt.show()

Primeros 8 autovalores observados: [9.02  2.743 1.707 1.505 1.158 0.976 0.916 0.829]
Percentil 95 aleatorio: [1.799 1.66  1.574 1.501 1.435 1.376 1.328 1.274]
Media aleatoria: [1.694 1.59  1.511 1.445 1.385 1.331 1.281 1.232]
Factores retenidos por percentil 95: 4
Factores retenidos por media aleatoria: 4
Cuarto autovalor observado = 1.505; percentil 95 aleatorio = 1.501; media aleatoria = 1.445


**Figura: análisis paralelo y gráfico de sedimentación.** La curva observada se compara con autovalores aleatorios. La decisión de retener cuatro factores queda respaldada porque los primeros cuatro autovalores observados superan la referencia aleatoria, aunque el cuarto factor es una decisión ajustada bajo el percentil 95.

## 4. Solución de cuatro factores

Con cuatro factores retenidos, se estima un análisis factorial exploratorio de factores comunes mediante máxima verosimilitud y rotación oblicua Oblimin. La extracción por máxima verosimilitud se utiliza porque el objetivo es modelar la varianza común asociada a factores latentes. La rotación Oblimin se usa porque la teoría de Blend permite que las cuatro características se correlacionen entre sí.

También se estima una solución Varimax como contraste. Varimax es una rotación ortogonal: fuerza correlaciones nulas entre factores. Por eso puede ser útil como análisis de robustez, pero no se adopta como solución principal si los factores muestran correlaciones relevantes o si conceptualmente no se espera independencia entre dimensiones.

Además de las cargas factoriales, se reporta la varianza explicada por cada factor. La varianza acumulada con Oblimin es cercana al 39%, lo que es modesto, pero habitual en escalas actitudinales con ítems psicológicos y contenido heterogéneo. Por eso la conclusión debe ser de apoyo general, aunque no completo, a la teoría.


In [12]:
fa=FactorAnalyzer(n_factors=4,rotation='oblimin',method='ml'); fa.fit(X)
L=pd.DataFrame(fa.loadings_,index=items,columns=['F1','F2','F3','F4'])
print(L.round(3).to_string())
print('\nVarianza por factor:')
varianza = pd.DataFrame(np.array(fa.get_factor_variance()).T,columns=['SS cargas','Proporción','Acumulada'],index=['F1','F2','F3','F4'])
print(varianza.round(4).to_string())
print('\nCorrelaciones factoriales:')
print(pd.DataFrame(fa.phi_,index=L.columns,columns=L.columns).round(3).to_string())

        F1     F2     F3     F4
q1   0.401 -0.140  0.285  0.340
q2   0.097 -0.329  0.357  0.272
q3   0.116  0.293 -0.048  0.508
q4   0.125  0.165  0.137  0.510
q5   0.056  0.055  0.669 -0.131
q6   0.160 -0.347  0.161  0.264
q7   0.139  0.452 -0.049  0.299
q8   0.575  0.222  0.036  0.242
q9   0.437 -0.114  0.258  0.372
q10  0.417 -0.069  0.178  0.044
q11  0.266  0.457  0.200  0.102
q12 -0.011  0.133  0.269 -0.164
q13  0.560  0.082 -0.033  0.209
q14  0.757  0.028  0.179 -0.337
q15  0.361 -0.017  0.137  0.184
q16  0.741  0.215 -0.024 -0.200
q17  0.788 -0.009 -0.034  0.136
q18  0.232 -0.322  0.404  0.142
q19 -0.055  0.568 -0.177  0.078
q20  0.035  0.450  0.052  0.199
q21  0.471  0.165 -0.040  0.270
q22  0.857 -0.074 -0.021  0.113
q23 -0.094  0.042  0.700  0.066
q24  0.097  0.284  0.127  0.481
q25  0.101  0.533  0.131  0.082
q26  0.278  0.593  0.020 -0.072
q27 -0.020  0.612  0.382  0.068
q28  0.012  0.211  0.533 -0.016

Varianza por factor:
    SS cargas  Proporción  Acumulada
F1     4.3219

### Comparación con Varimax como contraste

La solución Varimax se estima solo para evaluar robustez. Si la interpretación mejora bajo una rotación ortogonal, podría considerarse; si los factores están correlacionados y la estructura conceptual permite esas correlaciones, Oblimin sigue siendo preferible.


In [7]:
fa_varimax = FactorAnalyzer(n_factors=4, rotation='varimax', method='ml')
fa_varimax.fit(X)
L_varimax = pd.DataFrame(fa_varimax.loadings_, index=items, columns=['F1','F2','F3','F4'])
varimax_var = pd.DataFrame(
    np.array(fa_varimax.get_factor_variance()).T,
    columns=['SS cargas','Proporción','Acumulada'],
    index=['F1','F2','F3','F4']
)
print('Matriz rotada Varimax (contraste):')
print(L_varimax.round(3).to_string())
print('\nVarianza por factor con Varimax:')
print(varimax_var.round(4).to_string())
print(f'\nVarianza acumulada Oblimin = {varianza.loc["F4","Acumulada"]:.4f}')
print(f'Varianza acumulada Varimax = {varimax_var.loc["F4","Acumulada"]:.4f}')
print(f'Correlación factorial máxima en Oblimin = {np.max(np.abs(fa.phi_[np.triu_indices(4,1)])):.3f}')


Matriz rotada Varimax (contraste):
        F1     F2     F3     F4
q1   0.257  0.355  0.602  0.155
q2  -0.024  0.066  0.571  0.129
q3   0.599  0.132  0.247 -0.063
q4   0.519  0.142  0.410  0.048
q5   0.051  0.162  0.274  0.602
q6  -0.047  0.089  0.487 -0.025
q7   0.600  0.190  0.036  0.044
q8   0.513  0.544  0.295  0.118
q9   0.306  0.387  0.611  0.138
q10  0.114  0.378  0.306  0.171
q11  0.542  0.346  0.075  0.317
q12  0.029  0.061 -0.030  0.301
q13  0.364  0.495  0.289  0.024
q14  0.047  0.709  0.111  0.350
q15  0.228  0.328  0.341  0.110
q16  0.262  0.696  0.007  0.206
q17  0.309  0.680  0.356  0.047
q18 -0.058  0.197  0.548  0.222
q19  0.480  0.031 -0.313  0.007
q20  0.515  0.117 -0.015  0.137
q21  0.445  0.430  0.261  0.017
q22  0.263  0.731  0.400  0.053
q23  0.128  0.026  0.385  0.550
q24  0.588  0.140  0.322  0.080
q25  0.534  0.206 -0.068  0.267
q26  0.523  0.362 -0.204  0.262
q27  0.582  0.152 -0.016  0.482
q28  0.225  0.129  0.196  0.504

Varianza por factor con Varimax:
   

La comparación muestra que Varimax también produce cuatro factores y una varianza acumulada mayor. Sin embargo, al imponer independencia entre factores redistribuye varias cargas y no respeta la correlación observada entre dimensiones. Por ello, Varimax se conserva como contraste y la interpretación principal se basa en Oblimin.


## 5. Interpretación y confiabilidad

La solución se interpreta considerando conjuntamente contenido y cargas:

- F1: entusiasmo por diseño experimental.
- F2: amor por la enseñanza.
- F3: ausencia de habilidades interpersonales.
- F4: amor por la estadística.

Se calcula alfa con conjuntos conceptualmente coherentes. Algunos ítems presentan cargas cruzadas o débiles, por lo que la teoría se evalúa en grado, no solo contando factores.

In [14]:
def alpha(d):
 k=d.shape[1]; return k/(k-1)*(1-d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
sets={
'Diseño experimental':['q8','q13','q14','q16','q17','q22'],
'Enseñanza':['q7','q10','q11','q19','q20','q25','q26'],
'Habilidades interpersonales deficientes':['q2','q5','q6','q12','q18','q23','q27','q28'],
'Estadística':['q1','q3','q4','q9','q15','q21','q24']}
for nombre,its in sets.items(): print(f'{nombre}: alpha={alpha(X[its]):.3f} ({len(its)} ítems)')

Diseño experimental: alpha=0.880 (6 ítems)
Enseñanza: alpha=0.747 (7 ítems)
Habilidades interpersonales deficientes: alpha=0.691 (8 ítems)
Estadística: alpha=0.830 (7 ítems)


In [15]:
# Correlaciones ítem-total corregidas por subescala teórica.
# Verifica que ningún ítem esté invertido o mal codificado dentro de su subescala:
# una correlación ítem-total negativa indicaría necesidad de recodificación antes del alfa.
for nombre, its in sets.items():
    sub = X[its]
    print(f'\n{nombre}:')
    for it in its:
        resto = sub.drop(columns=it).sum(axis=1)
        r = sub[it].corr(resto)
        marca = '  <-- NEGATIVA' if r < 0 else ''
        print(f'  {it}: r item-total corregida = {r:.3f}{marca}')


Diseño experimental:
  q8: r item-total corregida = 0.714
  q13: r item-total corregida = 0.611
  q14: r item-total corregida = 0.634
  q16: r item-total corregida = 0.678
  q17: r item-total corregida = 0.767
  q22: r item-total corregida = 0.783

Enseñanza:
  q7: r item-total corregida = 0.476
  q10: r item-total corregida = 0.248
  q11: r item-total corregida = 0.538
  q19: r item-total corregida = 0.360
  q20: r item-total corregida = 0.505
  q25: r item-total corregida = 0.568
  q26: r item-total corregida = 0.583

Habilidades interpersonales deficientes:
  q2: r item-total corregida = 0.382
  q5: r item-total corregida = 0.486
  q6: r item-total corregida = 0.261
  q12: r item-total corregida = 0.170
  q18: r item-total corregida = 0.446
  q23: r item-total corregida = 0.566
  q27: r item-total corregida = 0.333
  q28: r item-total corregida = 0.424

Estadística:
  q1: r item-total corregida = 0.636
  q3: r item-total corregida = 0.516
  q4: r item-total corregida = 0.539
  q9: 

## 6. Conclusión

Bartlett es significativo ($\chi^2(378)=3045.39$, $p<0.001$) y KMO=0.895, por lo que la matriz es adecuada. El análisis paralelo recomienda **cuatro factores** tanto con el percentil 95 (cuarto autovalor 1.505 contra umbral 1.501, decisión ajustada) como con la media de los autovalores aleatorios (umbral 1.445, decisión holgada), coincidiendo con Blend y con el respaldo de la expectativa teórica a priori. La solución rotada permite reconocer las cuatro áreas teóricas, especialmente diseño experimental y estadística/enseñanza.

Sin embargo, hay cargas cruzadas (por ejemplo q1, q9 y q27) e ítems sin carga factorial relevante o débiles (como q12), y la dimensión de habilidades interpersonales deficientes no queda completamente limpia. **La teoría de Blend recibe apoyo general respecto del número y contenido de factores, pero no apoyo perfecto a nivel de todos los ítems.** Se recomienda revisar o eliminar ítems ambiguos antes de una validación confirmatoria.